In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
sns.set_style('ticks')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 25
two_colors = ['#6384B3', '#F08743']

In [ ]:
def bernoulli_ci(values):
    import statsmodels.api as sm
    n = len(values)
    p = values.mean()
    conf_int = sm.stats.proportion_confint(p * n, n, method='normal')
    return conf_int
data = pd.read_pickle('./processed_data/samples_10000_model.pkl')

# Plot the distribution of inequality across groups.
plt.figure(figsize=(8,6))
sns.barplot(data, x='age_range', y='indicator', order=['18-23','24-30','31-40','41-50','51-60','60+'], errorbar=bernoulli_ci, palette=sns.color_palette("Blues", 6), capsize=0.2, edgecolor=sns.color_palette("Blues", 6))
plt.hlines(y=data['indicator'].mean(), xmin=-0.75, xmax=5.75, lw=5, color='gray', ls='--', alpha=0.5, label='Avg')
plt.ylabel('Ratio for Deep ICs')
plt.xlabel('Age')
plt.xlim(-0.75, 5.75)
plt.legend()
print( (data.loc[data['age_range']=='60+']['indicator'].mean()- data.loc[data['age_range']=='18-23']['indicator'].mean())/data['indicator'].mean() )
print(data.loc[data['age_range']=='18-23']['indicator'].mean())

In [ ]:
with open('./processed_data/count_explore_exploit.pkl', 'rb') as f:
    results  = pickle.load(f)

In [ ]:
lower_cc = data.loc[data['age_range']=='18-23']['indicator'].mean()
avg_cc = data['indicator'].mean()
most_area = [(ss[0], ss[1]) for ss in sorted(results,key=lambda x: x[2], reverse=True)[:5]]

In [ ]:
most_area # sigma=4, beta=0. 3,8

In [ ]:
results

In [ ]:
# Adjust beta.
df_list=[]
beta_data = pd.read_pickle('./processed_data/intervention_beta.pkl')
for mm in most_area:
    df_tmp = beta_data.loc[(beta_data['sigma']==mm[0])&(beta_data['alpha']==mm[1])]
    df_tmp['type'] = df_tmp.apply(lambda x: 'sigma={}, alpha={}'.format(mm[0], mm[1]), axis=1)
    df_list.append(df_tmp[['type','indicator','entropy','beta']])
df_list = pd.concat(df_list)
df_list['reduced_beta'] = 7 - df_list['beta']

In [ ]:
plt.figure(figsize=(6.5,6))
hue_order = ['sigma=7, alpha=2', 'sigma=7, alpha=3', 'sigma=4, alpha=0', 'sigma=3, alpha=8', 'sigma=2, alpha=8']
pp_colors = sns.color_palette("Blues", 3)+sns.color_palette("YlOrBr", 2)
sns.lineplot(df_list, x='reduced_beta', y='indicator', hue='type', errorbar=bernoulli_ci, err_style='bars', err_kws={'capsize':5}, marker='o', hue_order=hue_order, palette=pp_colors, legend=False)
plt.xlabel(r'Reduced $\beta$')
plt.ylabel('Ratio for Deep ICs')
plt.savefig('./figures/figure4/beta_1.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
plt.figure(figsize=(6.5,6))
hue_order = ['sigma=7, alpha=2', 'sigma=7, alpha=3', 'sigma=4, alpha=0', 'sigma=3, alpha=8', 'sigma=2, alpha=8']
pp_colors = sns.color_palette("Blues", 3)+sns.color_palette("YlOrBr", 2)
sns.lineplot(df_list, x='reduced_beta', y='entropy', hue='type', err_style='bars', err_kws={'capsize':5}, marker='o', hue_order=hue_order, palette=pp_colors, legend=False)
plt.xlabel(r'Reduced $\beta$')
plt.ylabel(r'$\tilde{s}$')
plt.savefig('./figures/figure4/beta_2.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
with open('./processed_data/intervention_beta_summary.pkl', 'rb') as f:
    results_list = pickle.load(f)

plt.figure(figsize=(6.5,6))
plt.plot(7-np.linspace(6,7,11) , (results_list-lower_cc)/avg_cc, marker='o', lw=2, markersize=7, color='#6384B3')
plt.vlines(x=0,ymin=-0.2, ymax=0.3, color='gray', lw=10, alpha=0.2)
plt.ylim(-0.2, 0.25)
plt.xlabel(r'Reduced $\beta$')
plt.ylabel('Disparity')
plt.savefig('./figures/figure4/beta_3.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
# Adjust sigma.
df_list=[]
beta_data = pd.read_pickle('./processed_data/intervention_beta.pkl')
beta_data = beta_data.loc[beta_data['beta']==7]
for mm in most_area:
    for kk in range(5):
        df_tmp = beta_data.loc[(beta_data['sigma']==min(mm[0]+kk,9))&(beta_data['alpha']==mm[1])]
        df_tmp['type'] = df_tmp.apply(lambda x: 'sigma={}, alpha={}'.format(mm[0], mm[1]), axis=1)
        df_tmp['increased_sigma'] = kk
        df_list.append(df_tmp[['type','indicator','entropy','increased_sigma']])
df_list = pd.concat(df_list)

In [ ]:
df_list

In [ ]:
plt.figure(figsize=(6.5,6))
hue_order = ['sigma=7, alpha=2', 'sigma=7, alpha=3', 'sigma=4, alpha=0', 'sigma=3, alpha=8', 'sigma=2, alpha=8']
pp_colors = sns.color_palette("Blues", 3)+sns.color_palette("YlOrBr", 2)
sns.lineplot(df_list, x='increased_sigma', y='indicator', hue='type', errorbar=bernoulli_ci, err_style='bars', err_kws={'capsize':5}, marker='o', hue_order=hue_order, palette=pp_colors, legend=False)
plt.xlabel(r'Increased $\sigma$')
plt.ylabel('Ratio for Deep ICs')
plt.savefig('./figures/figure4/sigma_1.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
plt.figure(figsize=(6.5,6))
hue_order = ['sigma=7, alpha=2', 'sigma=7, alpha=3', 'sigma=4, alpha=0', 'sigma=3, alpha=8', 'sigma=2, alpha=8']
pp_colors = sns.color_palette("Blues", 3)+sns.color_palette("YlOrBr", 2)
sns.lineplot(df_list, x='increased_sigma', y='entropy', hue='type', err_style='bars', err_kws={'capsize':5}, marker='o', hue_order=hue_order, palette=pp_colors, legend=False)
plt.xlabel(r'Increased $\sigma$')
plt.ylabel(r'$\tilde{s}$')
plt.savefig('./figures/figure4/sigma_2.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
with open('./processed_data/intervention_sigma_summary.pkl', 'rb') as f:
    results_list = pickle.load(f)
    results_list = np.array([results_list[i] for i in range(10)])
plt.figure(figsize=(6.5,6))
plt.plot(np.arange(10) , (results_list-lower_cc)/avg_cc, marker='o', lw=2, markersize=7, color='#6384B3')
plt.vlines(x=0,ymin=-0.2, ymax=0.3, color='gray', lw=10, alpha=0.2)
plt.ylim(-0.2, 0.25)
plt.xlabel(r'Increased $\sigma$')
plt.ylabel('Disparity')
plt.savefig('./figures/figure4/sigma_3.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
# Adjust beta.
df_list=[]
beta_data = pd.read_pickle('./processed_data/intervention_beta.pkl')
beta_data = beta_data.loc[beta_data['beta']==7]
for mm in most_area:
    for kk in range(5):
        df_tmp = beta_data.loc[(beta_data['sigma']==mm[0])&(beta_data['alpha']==max(mm[1]-kk, 0))]
        df_tmp['type'] = df_tmp.apply(lambda x: 'sigma={}, alpha={}'.format(mm[0], mm[1]), axis=1)
        df_tmp['decreased_alpha'] = kk
        df_list.append(df_tmp[['type','indicator','entropy','decreased_alpha']])
df_list = pd.concat(df_list)

In [ ]:
df_list

In [ ]:
plt.figure(figsize=(6.5,6))
hue_order = ['sigma=7, alpha=2', 'sigma=7, alpha=3', 'sigma=4, alpha=0', 'sigma=3, alpha=8', 'sigma=2, alpha=8']
pp_colors = sns.color_palette("Blues", 3)+sns.color_palette("YlOrBr", 2)
sns.lineplot(df_list, x='decreased_alpha', y='indicator', hue='type', errorbar=bernoulli_ci, err_style='bars', err_kws={'capsize':5}, marker='o', hue_order=hue_order, palette=pp_colors, legend=False)
plt.xlabel(r'Reduced $\alpha$')
plt.ylabel('Ratio for Deep ICs')
plt.savefig('./figures/figure4/alpha_1.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
plt.figure(figsize=(6.5,6))
hue_order = ['sigma=7, alpha=2', 'sigma=7, alpha=3', 'sigma=4, alpha=0', 'sigma=3, alpha=8', 'sigma=2, alpha=8']
pp_colors = sns.color_palette("Blues", 3)+sns.color_palette("YlOrBr", 2)
sns.lineplot(df_list, x='decreased_alpha', y='entropy', hue='type', err_style='bars', err_kws={'capsize':5}, marker='o', hue_order=hue_order, palette=pp_colors, legend=False)
plt.xlabel(r'Reduced $\alpha$')
plt.ylabel(r'$\tilde{s}$')
plt.savefig('./figures/figure4/alpha_2.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
with open('./processed_data/intervention_alpha_summary.pkl', 'rb') as f:
    results_list = pickle.load(f)
    results_list = np.array([results_list[i] for i in range(10)])
plt.figure(figsize=(6.5,6))
plt.plot(np.arange(10) , (results_list-lower_cc)/avg_cc, marker='o', lw=2, markersize=7, color='#6384B3')
plt.vlines(x=0,ymin=-0.2, ymax=0.3, color='gray', lw=10, alpha=0.2)
plt.ylim(-0.2, 0.25)
plt.xlabel(r'Reduced $\alpha$')
plt.ylabel('Disparity')
plt.savefig('./figures/figure4/alpha_3.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
results_list

# Reformat the results as heatmaps.

In [ ]:
with open('./processed_data/age_range_all_dist.pkl','rb') as f:
    results = pickle.load(f)
age_dist = results['age_dist']
# explore, exploit


In [ ]:
# Locate the mean behavioural profile of the oldest group.
explore_avg = 0
exploit_avg = 0
for i in range(10):
    for j in range(10):
        explore_avg += results['60+'][i,j]*i
        exploit_avg += results['60+'][i,j]*j


In [ ]:
explore_avg, exploit_avg

In [ ]:
np.unravel_index(np.argmax(results['60+']), results['60+'].shape)

In [ ]:
import pickle
with open('./processed_data/phase_diagram.pkl', 'rb') as f:
    mat = pickle.load(f)
# beta, alpha, sigma
criterion = 0.7190319683690323
alpha_list = np.arange(5.6, 6.8, 0.1228)
sigma_list = np.arange(0.018, 0.0212, 0.0003458)
beta_list = np.array([0,1,2,3,4,5,6, 7,8,9])


avg = 0.374
min_value = 0.232
max_value=0.569

In [ ]:
# beta, alpha, sigma
mat_tmp = mat[8,:,:]
fig1, ax1 = plt.subplots(1,1,figsize=(6,6))
fig2, ax2 = plt.subplots(1,1,figsize=(0.5,6))

sns.heatmap(mat_tmp.T, center=avg, vmin=min_value, vmax=max_value, cmap=sns.color_palette("vlag_r", as_cmap=True),
            cbar_kws={'label':'Ratio for Deep ICs'},linewidths=0.5, linecolor='gray', ax=ax1, cbar_ax=ax2).invert_yaxis()
# plt.xticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
# plt.yticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
ax1.set_xticks(np.arange(0,15,5),['{}%'.format((i)*10) for i in np.arange(0,15,5)])
ax1.set_yticks(np.arange(5,15,5),['{}%'.format((i)*10) for i in np.arange(5,15,5)])
ax1.set_ylabel('Degree of Exploration')
ax1.set_xlabel('Degree of Exploitation')



# Get the current axes and show the outer spines.
ax = ax1

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1)  # Set the spine width.

# plt.savefig('./figures/figure3/heatmap_model.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

# fig1.savefig('./figures/figure4/beta_6.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
# fig2.savefig('./figures/figure4/colorbar.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:

# beta, alpha, sigma
mat_tmp = mat[6,:,:]
fig1, ax1 = plt.subplots(1,1,figsize=(6,6))
fig2, ax2 = plt.subplots(1,1,figsize=(0.5,6))

sns.heatmap(mat_tmp.T, center=avg, vmin=min_value, vmax=max_value, cmap=sns.color_palette("vlag_r", as_cmap=True),
            cbar_kws={'label':'Ratio for Deep ICs'},linewidths=0.5, linecolor='gray', ax=ax1, cbar_ax=ax2).invert_yaxis()
# plt.xticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
# plt.yticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
ax1.set_xticks(np.arange(0,15,5),['{}%'.format((i)*10) for i in np.arange(0,15,5)])
ax1.set_yticks(np.arange(5,15,5),['{}%'.format((i)*10) for i in np.arange(5,15,5)])
ax1.set_ylabel('Degree of Exploration')
ax1.set_xlabel('Degree of Exploitation')



# Get the current axes and show the outer spines.
ax = ax1

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1)  # Set the spine width.

# plt.savefig('./figures/figure3/heatmap_model.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

fig1.savefig('./figures/figure4/beta_6.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
fig2.savefig('./figures/figure4/colorbar.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
# beta, alpha, sigma
mat_tmp = mat[:,:,3]
plt.figure(figsize=(6,6))
sns.heatmap(mat_tmp.T, center=avg, vmin=min_value, vmax=max_value, cmap=sns.color_palette("vlag_r", as_cmap=True),
            cbar=False,linewidths=0.5, linecolor='gray').invert_yaxis()
# plt.xticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
# plt.yticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
plt.xticks(np.arange(0,15,5),['{}%'.format((i)*10) for i in np.arange(0,15,5)])
plt.yticks(np.arange(5,15,5),['{}%'.format((i)*10) for i in np.arange(5,15,5)])
plt.ylabel(r'Degree of Exploration')
plt.xlabel(r'Degree of Similarity-based Matching')



# Get the current axes and show the outer spines.
ax = plt.gca()

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1)  # Set the spine width.

plt.savefig('./figures/figure4/sigma_3.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
# beta, alpha, sigma
mat_tmp = mat[:,9,:]
plt.figure(figsize=(6,6))
sns.heatmap(mat_tmp.T, center=avg, vmin=min_value, vmax=max_value, cmap=sns.color_palette("vlag_r", as_cmap=True),
            cbar=False, linewidths=0.5, linecolor='gray').invert_yaxis()
# plt.xticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
# plt.yticks(np.arange(0,10)+0.5,['{}%'.format(i*10) for i in np.arange(1,11)])
plt.xticks(np.arange(0,15,5),['{}%'.format((i)*10) for i in np.arange(0,15,5)])
plt.yticks(np.arange(5,15,5),['{}%'.format((i)*10) for i in np.arange(5,15,5)])
plt.ylabel('Degree of Exploitation')
plt.xlabel('Degree of Simlarity-based Matching')



# Get the current axes and show the outer spines.
ax = plt.gca()

# Show all spines.
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1)  # Set the spine width.

plt.savefig('./figures/figure4/alpha_9.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
# Measure the reduction in information-cocoon inequality.
data_beta = pd.read_pickle('./processed_data/varying_beta.pkl')
data_alpha = pd.read_pickle('./processed_data/varying_alpha.pkl')
data_sigma = pd.read_pickle('./processed_data/varying_sigma.pkl')
print(data_beta.shape)
print(data_alpha.shape)
print(data_sigma.shape)

In [ ]:
data_beta

In [ ]:
lower_cc, avg_cc 

In [ ]:
plt.figure(figsize=(6.5,6))
sns.lineplot(data_beta, x='reduced_beta', y='indicator', errorbar=bernoulli_ci, err_style='bars', err_kws={'capsize':7, 'linewidth':2, 'capthick': 2}, marker='o', linewidth=2, markersize=8, color='#6384B3', label='60+')
plt.hlines(y=lower_cc,xmin=0,xmax=1, lw=5, color='#F08743', ls='--', label='18-23')
plt.hlines(y=avg_cc,xmin=0,xmax=1, lw=5, color='gray', ls='--', alpha=0.2, label='Avg')
plt.xticks([i*0.1 for i in range(0,12,2)], ['{}%'.format(i*10) for i in range(0,12,2)])
plt.ylabel('Ratio for Deep ICs')
plt.xlabel(r'Reduced $\beta$')
plt.legend()
plt.savefig('./figures/figure4/varying_beta.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
plt.figure(figsize=(6.5,6))
sns.lineplot(data_alpha, x='reduced_alpha', y='indicator', errorbar=bernoulli_ci, err_style='bars', err_kws={'capsize':7, 'linewidth':2, 'capthick': 2}, marker='o', linewidth=2, markersize=8, color='#6384B3')
plt.hlines(y=avg_cc,xmin=0,xmax=10, lw=5, color='gray', ls='--', alpha=0.2)
plt.hlines(y=lower_cc,xmin=0,xmax=10, lw=5, color='#F08743', ls='--')
plt.xticks([i for i in range(0,12,2)], ['{}%'.format(i*10) for i in range(0,12,2)])
plt.ylabel('Ratio for Deep ICs')
plt.xlabel(r'Reduced $\alpha$')
plt.savefig('./figures/figure4/varying_alpha.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)

In [ ]:
plt.figure(figsize=(6.5,6))
sns.lineplot(data_sigma, x='increased_sigma', y='indicator', errorbar=bernoulli_ci, err_style='bars', err_kws={'capsize':7, 'linewidth':2, 'capthick': 2}, marker='o', linewidth=2, markersize=8, color='#6384B3')
plt.hlines(y=avg_cc,xmin=0,xmax=10, lw=5, color='gray', ls='--', alpha=0.2)
plt.hlines(y=lower_cc,xmin=0,xmax=10, lw=5, color='#F08743', ls='--')
plt.ylabel('Ratio for Deep ICs')
plt.xticks([i for i in range(0,12,2)], ['{}%'.format(i*10) for i in range(0,12,2)])
plt.xlabel(r'Increased $\sigma$')
plt.savefig('./figures/figure4/varying_sigma.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)